# A2.2 · The bootstrap problem

**Function A — Security Architecture & Platform → The Identity & Non-Human Identity Engineer**  ·  *Security of AI*

Builds on **[A2.1 · "Who is calling?"](https://spbreed.github.io/cyber-commons/lessons/A2.1.html)**.

| | |
|---|---|
| Open-source tooling | SPIFFE/SPIRE, kind |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


Every identity system has a bootstrap problem: to prove who you are, you need a
credential; to get a credential, you need to prove who you are.

For humans we solve it out of band — someone checks a passport on the first day.
For workloads, the answer has historically been **secret zero**: a long-lived
API key placed in an environment variable, a config file, or a Kubernetes
Secret. Everything else hangs off it.

Secret zero has three properties that make it the root of most cloud breaches:

- **It never expires.** The median rotation interval for a service-account key
  in the wild is "never".
- **Anything that can read it becomes the workload.** A file read, a log line,
  a core dump, a compromised sidecar. It is a bearer credential with no context.
- **Revoking it breaks everything at once**, because it is shared, so nobody
  revokes it.

The modern answer is **attestation**. SPIFFE/SPIRE (and cloud equivalents like
IRSA or managed identities) issue an identity based on *properties of where the
workload is running* — this node, this pod, this container image — verified by
a platform component. No secret is planted, because none is needed. The
credential is short-lived and reissued continuously.

For agents this matters more than for ordinary services, because agents are
spawned dynamically, often per task, and a static secret handed to a fleet of
ephemeral workers is the worst version of secret zero.

## 2 · Demo — secret zero, working exactly as designed

First, the thing that works. This is a normal service-account bootstrap; there is nothing broken about it yet.

In [ ]:
import hashlib, time
from dataclasses import dataclass, field

ENVIRONMENT = {                       # what the process can read
    "AGENT_API_KEY": "sk-live-7f3c9a2b8e1d4f6a0c5b3e9d",
    "PATH": "/usr/bin",
}
ISSUED_AT = time.time() - 400 * 86400   # planted 400 days ago, as is typical

@dataclass
class StaticCredential:
    value: str
    issued: float
    def authenticate(self):
        return {"identity": "triage-agent", "method": "bearer secret",
                "age_days": round((time.time() - self.issued) / 86400),
                "ok": True}

cred = StaticCredential(ENVIRONMENT["AGENT_API_KEY"], ISSUED_AT)
print("bootstrap via secret zero:", cred.authenticate())
print("→ it works. Every service in your estate does this today.")

## 3 · Where it breaks — three ways, all ordinary

None of these require a sophisticated attacker. They require a log statement, a debug endpoint, or a curious process on the same host.

In [ ]:
import re

def leaks(env):
    found = []
    # 1. anything that can read the process environment
    found.append(("process env read", env.get("AGENT_API_KEY")))
    # 2. a well-meaning debug log
    log_line = f"starting agent with config {env}"
    m = re.search(r"sk-live-[a-z0-9]+", log_line)
    found.append(("debug log line", m.group(0) if m else None))
    # 3. a crash dump / error report
    crash = {"env": env, "stack": "..."}
    found.append(("crash report", crash["env"].get("AGENT_API_KEY")))
    return found

for how, value in leaks(ENVIRONMENT):
    print(f"{how:22s} → {'LEAKED ' + value[:14] + '…' if value else 'safe'}")

print(f"\ncredential age: {cred.authenticate()['age_days']} days")
print("exposure window for every one of those leaks: the whole 400 days,")
print("because nothing about a static secret expires on its own.")

## 4 · The control — attestation instead of a planted secret

SPIRE issues an SVID (an X.509 or JWT identity document) to a workload after verifying *node attestation* (this is really node-7, confirmed by the cloud provider's instance identity document) and *workload attestation* (this really is the process running image X, confirmed by reading the kernel's view of the process).

The agent never holds a secret. It asks a local socket for an identity, and gets a short-lived one it did not have to keep.

In [ ]:
@dataclass
class Attestor:
    """Stands in for the SPIRE agent's node + workload attestation."""
    node_id: str
    trusted_images: set

    def attest(self, claimed_sa, image, pid_namespace):
        # These facts come from the platform, not from the workload's own claims.
        if image not in self.trusted_images:
            return None, f"image {image!r} is not an attested workload image"
        if pid_namespace != self.node_id:
            return None, f"process is not running on {self.node_id}"
        return (f"spiffe://corp/ns/prod/sa/{claimed_sa}", "attested")

@dataclass
class SVID:
    spiffe_id: str
    issued: float = field(default_factory=time.time)
    ttl: float = 300                         # SPIRE default is minutes, not months
    @property
    def age_days(self): return (time.time() - self.issued) / 86400
    @property
    def expired(self): return time.time() - self.issued > self.ttl

spire = Attestor("node-7", {"ghcr.io/corp/triage-agent@sha256:9f2c…"})

for sa, image, node in [
    ("triage-agent", "ghcr.io/corp/triage-agent@sha256:9f2c…", "node-7"),
    ("triage-agent", "ghcr.io/attacker/evil@sha256:dead…",     "node-7"),
    ("triage-agent", "ghcr.io/corp/triage-agent@sha256:9f2c…", "laptop-of-contractor"),
]:
    sid, why = spire.attest(sa, image, node)
    if sid:
        svid = SVID(sid)
        print(f"ISSUED   {sid}  ttl={svid.ttl:.0f}s")
    else:
        print(f"REFUSED  sa={sa} image={image[:34]}… — {why}")

## 5 · Verify — compare the exposure windows

In [ ]:
def exposure(cred_kind, ttl_seconds, leaked_at_day):
    """How long a leaked credential stays useful."""
    if ttl_seconds is None:
        return {"kind": cred_kind, "useful_for": "until someone rotates it",
                "typical": "never rotated", "window_seconds": float("inf")}
    return {"kind": cred_kind, "useful_for": f"{ttl_seconds:.0f}s",
            "typical": "auto-reissued continuously", "window_seconds": ttl_seconds}

for row in (exposure("secret zero (env var)", None, 12),
            exposure("SPIFFE SVID", 300, 12)):
    print(f"{row['kind']:26s} useful for {row['useful_for']:32s} ({row['typical']})")

static_window = 400 * 86400
svid_window = 300
print(f"\nratio: a leaked static key is useful "
      f"{static_window / svid_window:,.0f}× longer than a leaked SVID")
print("\nAnd the SVID names its own holder — a stolen one identifies the thief's")
print("workload, which a shared bearer secret can never do.")

## What you just proved

The static credential authenticates successfully and is 400 days old. All three leak paths expose it. SPIRE issues an SVID only for the attested image on the attested node, refusing the wrong image and the contractor's laptop. The final comparison shows a leaked static key is useful roughly 115,200× longer than a leaked 300-second SVID.

## Your turn

Find the oldest non-human credential in one production account and compute its age in days. Then ask what would break if you rotated it this afternoon. If nobody knows, that is the finding — and it is the same finding at every organisation that has not done this yet.

---

**Next → [A2.3 · Shadow Autonomy](https://spbreed.github.io/cyber-commons/lessons/A2.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A2.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A2.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*